Chapter 5 Exercise solutions

In [ ]:
# 检查本章及练习所需的第三方库版本（numpy、tiktoken、torch、tensorflow 等）
# 检查本章及练习所需的第三方库版本（numpy、tiktoken、torch、tensorflow 等）
from importlib.metadata import version

# 注意：这行导入与本章练习内容无关，后面代码也没有用到 inverse_dict，疑似多余/误加的导入；
# 若当前环境未安装 nltk 会导致本单元直接报错（风险项，未做改动，仅标注供确认）
# 注意：这行导入与本章练习内容无关，后面代码也没有用到 inverse_dict，疑似多余/误加的导入；
# 若当前环境未安装 nltk 会导致本单元直接报错（风险项，未做改动，仅标注供确认）
from nltk.langnames import inverse_dict

# tensorflow 是后面加载 OpenAI 官方发布的 GPT-2 预训练权重（TensorFlow checkpoint 格式）时需要用到的库
# tensorflow 是后面加载 OpenAI 官方发布的 GPT-2 预训练权重（TensorFlow checkpoint 格式）时需要用到的库
pkgs = ["numpy",
        "tiktoken",
        "torch",
        "tensorflow" # For OpenAI's pretrained weights
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

Exercise 5.1: Temperature-scaled softmax scores and sampling probabilities

In [ ]:
# 练习 5.1：带温度缩放的 softmax 与采样概率
import torch

# 一个玩具词表：词 -> id 的映射
vocab = {
    "closer": 0,
    "every": 1,
    "effort": 2,
    "forward": 3,
    "inches": 4,
    "moves": 5,
    "pizza": 6,
    "toward": 7,
    "you": 8,
}
# 反向映射：id -> 词，方便把采样得到的 id 转换回可读的词
inverse_vocab={v:k for k,v in vocab.items()}
# 假设这是模型针对“下一个词”输出的原始 logits（未归一化的分数）
next_token_logits=torch.tensor([4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79])
def print_sampled_tokens(probas):
    # 固定随机种子，保证多次运行的采样结果可复现
    torch.manual_seed(123)
    # 按给定的概率分布做 1000 次多项式（multinomial）采样，模拟“按概率随机选下一个词”的生成过程
    sampled=[torch.multinomial(probas,num_samples=1).item() for i in range(1_000)]
    # 统计词表中每个词被采样到的次数，用于观察分布形状
    sampled_ids=torch.bincount(torch.tensor(sampled))
    for i ,freq in enumerate(sampled_ids):
         print(f"{freq} x {inverse_vocab[i]}")

def softmax_with_temperature(logits,temperature):
    # 温度缩放：温度越大分布越平滑（采样更随机、更多样）；温度越小分布越尖锐（越接近贪心 argmax）
    scaled_logits=logits/temperature
    return torch.softmax(scaled_logits,dim=0)
temperatures = [1, 0.1, 5]  # Original, higher, and lower temperature
# 分别用三种温度计算缩放后的概率分布，便于下面对比不同温度下的采样效果
scaled_probas=[softmax_with_temperature(next_token_logits,T) for T in temperatures]

In [ ]:
# 依次遍历三种温度设置，打印各自的采样词频分布，直观对比温度对随机性的影响
# 依次遍历三种温度设置，打印各自的采样词频分布，直观对比温度对随机性的影响
for i, probas in enumerate(scaled_probas):
    print("\n\nTemperature:", temperatures[i])
    print_sampled_tokens(probas)

In [ ]:
# temperatures[2] 对应温度 = 5（较高温度，分布更平滑、更随机）
# temperatures[2] 对应温度 = 5（较高温度，分布更平滑、更随机）
temp5_idx = 2
# "pizza" 在词表中的 id
# "pizza" 在词表中的 id
pizza_idx = 6

# 查看温度为 5 时，"pizza"（语义上并不通顺的词）被采样到的概率
# 查看温度为 5 时，"pizza"（语义上并不通顺的词）被采样到的概率
scaled_probas[temp5_idx][pizza_idx]

Exercise 5.2: Different temperature and top-k settings
Both temperature and top-k settings have to be adjusted based on the individual LLM (a kind of trial and error process until it generates desirable outputs)
The desirable outcomes are also application-specific, though
Lower top-k and temperatures result in less random outcomes, which is desired when creating educational content, technical writing or question answering, data analyses, code generation, and so forth
Higher top-k and temperatures result in more diverse and random outputs, which is more desirable for brainstorming tasks, creative writing, and so forth

Exercise 5.3: Deterministic behavior in the decoding functions

In [ ]:
# 练习 5.3：解码函数中的确定性行为
# 练习 5.3：解码函数中的确定性行为
import  tiktoken
import  torch
from previous_chapters import GPTModel
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size
    "context_length": 256,       # Shortened context length (orig: 1024)
    "emb_dim": 768,       # Embedding dimension
    "n_heads": 12,        # Number of attention heads
    "n_layers": 12,       # Number of layers
    "drop_rate": 0.1,     # Dropout rate
    "qkv_bias": False     # Query-key-value bias
}
# 固定随机种子，保证初始化等操作可复现
# 固定随机种子，保证初始化等操作可复现
torch.manual_seed(123)
tokenizer=tiktoken.get_encoding("gpt2")
model = GPTModel(GPT_CONFIG_124M)
# 加载第 5 章训练后保存的模型权重；weights_only=True 更安全，只反序列化张量，不执行任意代码
# 加载第 5 章训练后保存的模型权重；weights_only=True 更安全，只反序列化张量，不执行任意代码
model.load_state_dict(torch.load("model.pth", weights_only=True))
# 切换到评估模式（关闭 dropout 等训练专属行为），保证生成结果是确定的
# 切换到评估模式（关闭 dropout 等训练专属行为），保证生成结果是确定的
model.eval();

In [ ]:
# generate：支持 top-k 采样和温度缩放的解码函数；text_to_token_ids/token_ids_to_text：文本与 token id 互转的工具函数
# generate_text_simple：第 4 章实现的、只用 argmax 贪心解码的简单生成函数，用于和 generate 做对比
# generate：支持 top-k 采样和温度缩放的解码函数；text_to_token_ids/token_ids_to_text：文本与 token id 互转的工具函数
# generate_text_simple：第 4 章实现的、只用 argmax 贪心解码的简单生成函数，用于和 generate 做对比
from gpt_generate import generate, text_to_token_ids, token_ids_to_text
from previous_chapters import generate_text_simple

In [ ]:
# 说明：generate_text_simple 每一步都用 argmax 选择概率最高的 token，没有任何随机性，
# 因此只要模型权重和输入不变，输出必然是确定且可复现的
# 说明：generate_text_simple 每一步都用 argmax 选择概率最高的 token，没有任何随机性，
# 因此只要模型权重和输入不变，输出必然是确定且可复现的
# Deterministic function that used torch.argmax
start_context = "Every effort moves you"
token_ids=generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context,tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M['context_length']
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
# 说明：generate() 在 top_k=None 且 temperature=0 时会退化为等价于 argmax 的贪心解码，
# 因此输出应与上一单元 generate_text_simple 的结果完全一致
# 说明：generate() 在 top_k=None 且 temperature=0 时会退化为等价于 argmax 的贪心解码，
# 因此输出应与上一单元 generate_text_simple 的结果完全一致
# Deterministic behavior: No top_k, no temperature scaling
token_ids=generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you",tokenizer),
    context_size=GPT_CONFIG_124M["context_length"],
    max_new_tokens=25,
    top_k=None,
    temperature=0
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
# 说明：这里改用浮点数 temperature=0.0（而非整数 0）再验证一次，
# 结果应与前两个单元一致，证明 0 和 0.0 触发的是同一条“贪心解码”分支
# 说明：这里改用浮点数 temperature=0.0（而非整数 0）再验证一次，
# 结果应与前两个单元一致，证明 0 和 0.0 触发的是同一条“贪心解码”分支
# Deterministic behavior: No top_k, no temperature scaling
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=None,
    temperature=0.0
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Exercise 5.4: Continued pretraining

In [ ]:
# 练习 5.4：继续预训练——从之前保存的“模型+优化器”checkpoint 恢复训练
# 练习 5.4：继续预训练——从之前保存的“模型+优化器”checkpoint 恢复训练
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
# 加载同时包含模型权重和优化器状态的 checkpoint：继续训练需要恢复优化器状态（如动量），
# 若只恢复模型权重、用全新优化器状态重新开始，训练效果会打折扣
# 加载同时包含模型权重和优化器状态的 checkpoint：继续训练需要恢复优化器状态（如动量），
# 若只恢复模型权重、用全新优化器状态重新开始，训练效果会打折扣
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

# 重新创建优化器结构后，再加载其保存的状态（一阶/二阶动量等），实现“无缝续训”
# 重新创建优化器结构后，再加载其保存的状态（一阶/二阶动量等），实现“无缝续训”
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# 切回训练模式（重新启用 dropout 等），为继续训练做准备
# 切回训练模式（重新启用 dropout 等），为继续训练做准备
model.train();

In [ ]:
# 准备继续预训练用的数据：下载文本、划分训练/验证集、构建 DataLoader
# 准备继续预训练用的数据：下载文本、划分训练/验证集、构建 DataLoader
import os
import requests
from previous_chapters import create_dataloader_v1


file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import urllib.request

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
"""
# 按 90/10 划分训练集与验证集
# 按 90/10 划分训练集与验证集
# Train/validation ratio
train_ratio=0.90
split_idx=int(train_ratio*len(text_data))
train_data=text_data[:split_idx]
val_data=text_data[split_idx:]

# 固定随机种子，保证 DataLoader 打乱（shuffle）顺序可复现
# 固定随机种子，保证 DataLoader 打乱（shuffle）顺序可复现
torch.manual_seed(123)
train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# 使用第 5 章实现的简单训练循环，从恢复的 checkpoint 继续训练 1 个 epoch
# 使用第 5 章实现的简单训练循环，从恢复的 checkpoint 继续训练 1 个 epoch
from gpt_train import train_model_simple
# 只再训练 1 个 epoch，做“继续预训练”的演示
# 只再训练 1 个 epoch，做“继续预训练”的演示
num_epochs=1
train_losses,val_losses,tokens_seen=train_model_simple(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=5,
    eval_iter=5,
    start_context="Every effort moves you",
    tokenizer=tokenizer
)

Exercise 5.5: Training and validation set losses of the pretrained model

In [ ]:
# 练习 5.5：对比使用 OpenAI 官方预训练权重前后的训练/验证损失
# 练习 5.5：对比使用 OpenAI 官方预训练权重前后的训练/验证损失
import  tiktoken
import torch
from previous_chapters import GPTModel
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}
# 固定随机种子，保证后续（未加载预训练权重时）模型初始化可复现
# 固定随机种子，保证后续（未加载预训练权重时）模型初始化可复现
torch.manual_seed(123)
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# 下载并加载 OpenAI 官方发布的 GPT-2 权重文件（TensorFlow checkpoint 格式）及超参数
# 下载并加载 OpenAI 官方发布的 GPT-2 权重文件（TensorFlow checkpoint 格式）及超参数
from gpt_download import download_and_load_gpt2
# settings：模型超参数（如 n_layer/n_head 等）；params：权重字典（numpy 数组），后续需要搬运到我们自己的 GPTModel 里
# settings：模型超参数（如 n_layer/n_head 等）；params：权重字典（numpy 数组），后续需要搬运到我们自己的 GPTModel 里
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
# 官方 GPT-2 各规格模型的结构参数（嵌入维度、层数、注意力头数）
# 官方 GPT-2 各规格模型的结构参数（嵌入维度、层数、注意力头数）
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
# 选用与下载权重匹配的规格（124M / small）
# 选用与下载权重匹配的规格（124M / small）
model_name="gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
# 官方 GPT-2 的上下文长度是 1024（比本书示例用的 256 更长）；
# qkv_bias=True 是因为 OpenAI 的官方权重里 Q/K/V 线性层带有偏置项，必须保持结构一致才能正确加载权重
# 官方 GPT-2 的上下文长度是 1024（比本书示例用的 256 更长）；
# qkv_bias=True 是因为 OpenAI 的官方权重里 Q/K/V 线性层带有偏置项，必须保持结构一致才能正确加载权重
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

# 用匹配官方结构的配置新建一个空模型，稍后把下载到的权重加载进来；eval() 关闭 dropout 便于确定性生成
# 用匹配官方结构的配置新建一个空模型，稍后把下载到的权重加载进来；eval() 关闭 dropout 便于确定性生成
gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [ ]:
# load_weights_into_gpt：把 OpenAI 官方权重（numpy 数组）按名称/形状对应地拷贝进我们自己实现的 GPTModel
# load_weights_into_gpt：把 OpenAI 官方权重（numpy 数组）按名称/形状对应地拷贝进我们自己实现的 GPTModel
from gpt_generate import load_weights_into_gpt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 执行权重搬运：搬运完成后 gpt 就等价于官方预训练的 GPT-2（124M）
# 执行权重搬运：搬运完成后 gpt 就等价于官方预训练的 GPT-2（124M）
load_weights_into_gpt(gpt, params)
# 将模型移动到可用设备（GPU 优先，否则 CPU）
# 将模型移动到可用设备（GPU 优先，否则 CPU）
gpt.to(device);

In [ ]:
# 为评估“加载了官方预训练权重的模型”准备数据（与练习 5.4 中的数据加载单元结构相同，但属于本练习独立的一份流程）
# 为评估“加载了官方预训练权重的模型”准备数据（与练习 5.4 中的数据加载单元结构相同，但属于本练习独立的一份流程）
import os
import urllib.request
from previous_chapters import create_dataloader_v1


file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


# 同样按 90/10 划分训练集与验证集
# 同样按 90/10 划分训练集与验证集
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


# 固定随机种子，保证 DataLoader 打乱顺序可复现
# 固定随机种子，保证 DataLoader 打乱顺序可复现
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# calc_loss_loader：遍历 DataLoader，计算整个数据集上的平均交叉熵损失
# calc_loss_loader：遍历 DataLoader，计算整个数据集上的平均交叉熵损失
from gpt_train import calc_loss_loader

torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader
# 用加载了官方预训练权重的 gpt 模型分别计算训练/验证损失，
# 预期比随机初始化或只训练了很少数据的模型损失低得多，体现预训练权重的价值
# 用加载了官方预训练权重的 gpt 模型分别计算训练/验证损失，
# 预期比随机初始化或只训练了很少数据的模型损失低得多，体现预训练权重的价值
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
# 换成参数量最大的 gpt2-xl（1558M），下载对应的官方权重
# 换成参数量最大的 gpt2-xl（1558M），下载对应的官方权重
settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")

model_name = "gpt2-xl (1558M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

# 按 1558M 规格新建模型并加载对应的预训练权重
# 按 1558M 规格新建模型并加载对应的预训练权重
gpt = GPTModel(NEW_CONFIG)
gpt.eval()

load_weights_into_gpt(gpt, params)
gpt.to(device)

# 用同样的数据评估更大模型的训练/验证损失，预期比 124M 模型更低（模型越大、预训练效果通常越好）
# 用同样的数据评估更大模型的训练/验证损失，预期比 124M 模型更低（模型越大、预训练效果通常越好）
torch.manual_seed(123)
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Exercise 5.6: Trying larger models

In [ ]:
# 练习 5.6：尝试更大的预训练模型（gpt2-xl, 1558M）并用它做文本生成
# 练习 5.6：尝试更大的预训练模型（gpt2-xl, 1558M）并用它做文本生成
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}


# 这里不需要 manual_seed 初始化随机模型，因为后面会直接加载官方预训练权重，不使用随机初始化的参数
# 这里不需要 manual_seed 初始化随机模型，因为后面会直接加载官方预训练权重，不使用随机初始化的参数
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# 一次性完成：构建 1558M 规格的模型结构 -> 下载官方权重 -> 把权重加载进模型
# 一次性完成：构建 1558M 规格的模型结构 -> 下载官方权重 -> 把权重加载进模型
from gpt_download import download_and_load_gpt2
from gpt_generate import load_weights_into_gpt


# 官方 GPT-2 各规格模型的结构参数
# 官方 GPT-2 各规格模型的结构参数
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# 本次选用参数量最大的 gpt2-xl 规格
# 本次选用参数量最大的 gpt2-xl 规格
model_name = "gpt2-xl (1558M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

# 按 xl 规格新建空模型，eval() 关闭 dropout 便于确定性生成
# 按 xl 规格新建空模型，eval() 关闭 dropout 便于确定性生成
gpt = GPTModel(NEW_CONFIG)
gpt.eval()

# 下载 1558M 官方权重并搬运进模型，完成后 gpt 即为官方预训练的 gpt2-xl
# 下载 1558M 官方权重并搬运进模型，完成后 gpt 即为官方预训练的 gpt2-xl
settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")
load_weights_into_gpt(gpt, params)

In [ ]:
# 导入支持 top-k / 温度采样的生成函数，以及文本与 token id 互转的工具函数
# 导入支持 top-k / 温度采样的生成函数，以及文本与 token id 互转的工具函数
from gpt_generate import generate, text_to_token_ids, token_ids_to_text

In [ ]:
# 固定随机种子，保证带随机采样的生成过程也可复现
# 固定随机种子，保证带随机采样的生成过程也可复现
torch.manual_seed(123)

# 用 gpt2-xl 生成文本：top_k=50 只在概率最高的 50 个候选词中采样，temperature=1.5 让分布更平滑、输出更多样（更“有创意”）
# 用 gpt2-xl 生成文本：top_k=50 只在概率最高的 50 个候选词中采样，temperature=1.5 让分布更平滑、输出更多样（更“有创意”）
token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))